<a href="https://colab.research.google.com/github/Muneebshah1192/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

One row represents one content item for one client on one report date. I use the March 2026 observation window, from 2026-03-01 through 2026-03-31. In the March slice, I observed 9,841,378 rows, and the observed dates span the full month.

In [1]:
# ML-04 — Data setup
import os
import duckdb
import pandas as pd

# Confirm HF token is available through Colab Secrets
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found. Add it in Colab Secrets."

print("HF token loaded successfully.")

HF token loaded successfully.


In [2]:
# DuckDB connection
con = duckdb.connect()

print("DuckDB is ready.")

DuckDB is ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
# Show tables available in the DuckDB connection
con.sql("SHOW TABLES").df()

,name


In [8]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [11]:
fact_daily = TABLES["fact_daily"]

con.sql(f"""
DESCRIBE SELECT *
FROM {fact_daily}
LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [13]:
# Section 1 — Check whether client + content + date uniquely identifies a row

con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 20
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


In [12]:
con.sql(f"""
SELECT *
FROM {fact_daily}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [14]:
# Section 1 — March 2026 row count and date span

con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [15]:
con.sql(f"""
SELECT
    report_date,
    COUNT(*) AS rows
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY report_date
ORDER BY report_date
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,rows
0,2026-03-01,275874
1,2026-03-02,276269
2,2026-03-03,311676
3,2026-03-04,311675
4,2026-03-05,311676
5,2026-03-06,312187
6,2026-03-07,312387
7,2026-03-08,313374
8,2026-03-09,313874
9,2026-03-10,314047


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features

- `gsc_impressions` — historical search visibility available at the observation point.
- `gsc_clicks` — historical search traffic available at the observation point.
- `gsc_avg_position` — historical search ranking information available at the observation point.
- `ga4_sessions` — historical site-session activity available at the observation point.
- `sessions_ai` — historical sessions attributed to AI sources available at the observation point.

### Label

The label will be selected after verifying the available outcome definition and prediction window. I will not use a same-window outcome as a future label without checking the warehouse's time-window logic.

### Context

- `report_date` — identifies when the observation was recorded.
- `client_hash_id` — identifies the client entity without exposing a client name.
- `content_hash_id` — identifies the content entity without exposing a URL.
- `month` — identifies the observation month.
- `client_has_gsc` — indicates whether the client has GSC coverage.
- `client_has_ga4` — indicates whether the client has GA4 coverage.
- `gsc_data_available` — indicates whether GSC data is available for the observation.
- `ga4_data_available` — indicates whether GA4 data is available for the observation.

### Excluded

- `gsc_sum_position` — excluded because `gsc_avg_position` already provides a more directly interpretable position feature for this small feature set.
- `ga4_pageviews`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec` — excluded to keep the first feature frame small and avoid adding highly related engagement measures.
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid` — excluded from the first five-feature frame to avoid expanding the feature set beyond the required five features.
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` — excluded because `sessions_ai` is used as the aggregate AI-traffic feature instead.
- `scroll_events` — excluded from the first feature frame to keep the feature set limited to five variables.

In [16]:
con.sql(f"""
DESCRIBE SELECT *
FROM {fact_daily}
LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [17]:
fact_query = TABLES["fact_query_90d"]

con.sql(f"""
DESCRIBE SELECT *
FROM {fact_query}
LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [18]:
con.sql(f"""
SELECT *
FROM {fact_query}
LIMIT 5
""").df()

,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


## 2. Fields: feature / label / context / excluded

### Features

- `gsc_impressions` — observed search impressions available at the observation point.
- `gsc_clicks` — observed search clicks available at the observation point.
- `gsc_avg_position` — observed average search position available at the observation point.
- `ga4_sessions` — observed sessions available at the observation point.
- `sessions_ai` — observed AI-attributed sessions available at the observation point.

### Label

For this contract, the label is a future outcome derived from a later observation window. Same-window outcome fields are not used as labels because they would not represent a future prediction target.

### Context

- `report_date` — identifies the observation date.
- `client_hash_id` — identifies the client without exposing a client name.
- `content_hash_id` — identifies the content without exposing a URL.
- `month` — identifies the observation month.
- `client_has_gsc` — indicates whether GSC is available for the client.
- `client_has_ga4` — indicates whether GA4 is available for the client.
- `gsc_data_available` — indicates whether GSC data is available for the observation.
- `ga4_data_available` — indicates whether GA4 data is available for the observation.

### Excluded

- `gsc_sum_position` — excluded because `gsc_avg_position` is the selected position feature.
- `ga4_pageviews`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec` — excluded to keep the first feature frame limited to five features.
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid` — excluded from the first five-feature frame.
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` — excluded because `sessions_ai` is used as the aggregate AI-traffic feature.
- `scroll_events` — excluded to keep the feature frame limited to five features.
- `fact_query_90d` fields — excluded from the initial feature frame because their rolling 90-day and windowed measurements require careful temporal alignment to avoid leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
# Query 1 — Verify the claimed grain
grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 20
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


In [20]:
# Query 2 — March row count and date span
march_window = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

march_window

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [22]:
# Query 3 — GSC availability
gsc_availability = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

gsc_availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows
0,9841378,3611061


In [23]:
gsc_availability = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

gsc_availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows
0,9841378,3611061


In [24]:
gsc_availability = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

gsc_availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,gsc_available_rows
0,9841378,3611061


In [25]:
# Check availability of the five proposed features in March 2026

feature_availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(gsc_impressions) AS gsc_impressions_non_null,
    COUNT(gsc_clicks) AS gsc_clicks_non_null,
    COUNT(gsc_avg_position) AS gsc_avg_position_non_null,
    COUNT(ga4_sessions) AS ga4_sessions_non_null,
    COUNT(sessions_ai) AS sessions_ai_non_null

FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

feature_availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_impressions_non_null,gsc_clicks_non_null,gsc_avg_position_non_null,ga4_sessions_non_null,sessions_ai_non_null
0,9841378,9841378,9841378,3611061,6822637,6822637


In [26]:
five_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    sessions_ai
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
LIMIT 1000
""").df()

five_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_ai
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>


In [27]:
# Five-feature frame — March 2026

five_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    sessions_ai
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
LIMIT 1000
""").df()

five_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_ai
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>


In [28]:
con.sql(f"""
SELECT
    MIN(window_start) AS earliest_window_start,
    MAX(window_start) AS latest_window_start,
    MIN(window_end) AS earliest_window_end,
    MAX(window_end) AS latest_window_end,
    COUNT(*) AS total_rows
FROM {fact_query}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_window_start,latest_window_start,earliest_window_end,latest_window_end,total_rows
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30,2414248


In [29]:
con.sql(f"""
SELECT
    window_start,
    window_end,
    COUNT(*) AS rows
FROM {fact_query}
GROUP BY
    window_start,
    window_end
ORDER BY
    window_start
LIMIT 20
""").df()

,window_start,window_end,rows
0,2026-04-02,2026-06-30,2414248


In [30]:
# Future outcome: did this content receive at least one click
# during the sealed future window (2026-04-02 to 2026-06-30)?

future_outcomes = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(clicks_90d) AS future_clicks_90d,
    CASE
        WHEN SUM(clicks_90d) > 0 THEN TRUE
        ELSE FALSE
    END AS future_click_label
FROM {fact_query}
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

future_outcomes.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,future_clicks_90d,future_click_label
0,client_08a6a72ff48e62c0,content_5f6556f5a53e47f0,0.0,False
1,client_08a6a72ff48e62c0,content_5f6fae04728d32ab,0.0,False
2,client_08a6a72ff48e62c0,content_5f71205e0b46f70a,0.0,False
3,client_08a6a72ff48e62c0,content_5f716493f7989b45,0.0,False
4,client_08a6a72ff48e62c0,content_5f8b67a6b0494e15,0.0,False


In [31]:
future_outcomes["future_click_label"].value_counts(dropna=False)

,count
future_click_label,
False,94494
True,39358


- `future_clicks_90d` — excluded from the honest feature set because it is measured in the future outcome window and therefore would leak information about the label.

### Label

- `future_click_label` — a binary future outcome indicating whether the content received at least one click during the sealed future window from 2026-04-02 through 2026-06-30. This label is not available at the March 2026 decision moment.

In [32]:
# Build the honest modeling dataset:
# March 2026 features + future click label

honest_df = con.sql(f"""
WITH march_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        sessions_ai
    FROM {fact_daily}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
),

future_labels AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN SUM(clicks_90d) > 0 THEN 1
            ELSE 0
        END AS future_click_label
    FROM {fact_query}
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.*,
    f.future_click_label
FROM march_features m
INNER JOIN future_labels f
    ON m.client_hash_id = f.client_hash_id
   AND m.content_hash_id = f.content_hash_id
""").df()

print("Rows:", len(honest_df))
honest_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3224352


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_ai,future_click_label
0,client_08a6a72ff48e62c0,content_447b93d5ff670356,12,0,12.333333,<NA>,<NA>,0
1,client_08a6a72ff48e62c0,content_4483e354a2992497,0,0,NaN,<NA>,<NA>,0
2,client_08a6a72ff48e62c0,content_4489a53aa451dca0,0,0,NaN,<NA>,<NA>,0
3,client_08a6a72ff48e62c0,content_44a8fa878cc63e32,1,0,11.000000,<NA>,<NA>,0
4,client_08a6a72ff48e62c0,content_44afc7eca6a713b7,3,0,54.000000,<NA>,<NA>,0


In [33]:
honest_df.isna().sum()

,0
client_hash_id,0
content_hash_id,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,509779
ga4_sessions,1427719
sessions_ai,1427719
future_click_label,0


In [34]:
honest_df.shape

(3224352, 8)

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Features and label
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_ai"
]

X = honest_df[feature_cols]
y = honest_df["future_click_label"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Learn imputation values ONLY from training data
imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training label rate:", y_train.mean())
print("Test label rate:", y_test.mean())

Training rows: 2579481
Test rows: 644871
Training label rate: 0.2924231657453573
Test label rate: 0.2924228256504014


In [36]:
# Honest baseline model

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

honest_model.fit(X_train_imp, y_train)

honest_prob = honest_model.predict_proba(X_test_imp)[:, 1]

honest_auc = roc_auc_score(y_test, honest_prob)

print(f"Honest ROC-AUC: {honest_auc:.4f}")

Honest ROC-AUC: 0.7512


In [37]:
# Deliberate leakage experiment:
# Add future_clicks_90d as a feature on purpose.

leak_df = con.sql(f"""
WITH march_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        sessions_ai
    FROM {fact_daily}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
),

future_outcomes AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(clicks_90d) AS future_clicks_90d,
        CASE
            WHEN SUM(clicks_90d) > 0 THEN 1
            ELSE 0
        END AS future_click_label
    FROM {fact_query}
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.*,
    f.future_clicks_90d,
    f.future_click_label
FROM march_features m
INNER JOIN future_outcomes f
    ON m.client_hash_id = f.client_hash_id
   AND m.content_hash_id = f.content_hash_id
""").df()

leak_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_ai,future_clicks_90d,future_click_label
0,client_08a6a72ff48e62c0,content_447b93d5ff670356,3,0,0.000000,<NA>,<NA>,0.0,0
1,client_08a6a72ff48e62c0,content_4483e354a2992497,6,0,4.333333,<NA>,<NA>,0.0,0
2,client_08a6a72ff48e62c0,content_4489a53aa451dca0,3,0,4.666667,<NA>,<NA>,0.0,0
3,client_08a6a72ff48e62c0,content_44a8fa878cc63e32,3,0,62.666667,<NA>,<NA>,0.0,0
4,client_08a6a72ff48e62c0,content_44afc7eca6a713b7,2,0,77.000000,<NA>,<NA>,0.0,0


In [38]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

leak_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_ai",
    "future_clicks_90d"  # 🚨 DELIBERATE LEAK
]

X_leak = leak_df[leak_features]
y_leak = leak_df["future_click_label"]

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leak,
    y_leak,
    test_size=0.20,
    random_state=42,
    stratify=y_leak
)

leak_imputer = SimpleImputer(strategy="median")

X_train_leak_imp = leak_imputer.fit_transform(X_train_leak)
X_test_leak_imp = leak_imputer.transform(X_test_leak)

leak_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

leak_model.fit(X_train_leak_imp, y_train_leak)

leak_prob = leak_model.predict_proba(X_test_leak_imp)[:, 1]

leak_auc = roc_auc_score(y_test_leak, leak_prob)

print(f"Honest ROC-AUC: {honest_auc:.4f}")
print(f"Leaked ROC-AUC: {leak_auc:.4f}")

Honest ROC-AUC: 0.7512
Leaked ROC-AUC: 1.0000


### Deliberate leakage experiment

The honest five-feature model achieved a ROC-AUC of **0.7512**.

I then deliberately added `future_clicks_90d` as a feature. This field is calculated from the future outcome window of **2026-04-02 through 2026-06-30**, while the decision/feature window is March 2026.

The leaked model achieved a ROC-AUC of **1.0000**, compared with **0.7512** for the honest model.

This jump is caused by label leakage: `future_clicks_90d` contains information from the same future period used to construct `future_click_label`. Therefore, the leaked score is not a valid estimate of real-world predictive performance.

I removed `future_clicks_90d` from the final feature set and retained the honest ROC-AUC of **0.7512**.

### Deliberate leakage experiment

The honest five-feature model achieved a ROC-AUC of **0.7512**.

I then deliberately added `future_clicks_90d` as a feature. This field is calculated from the future outcome window of **2026-04-02 through 2026-06-30**, while the decision/feature window is March 2026.

The leaked model achieved a ROC-AUC of **1.0000**, compared with **0.7512** for the honest model.

This jump is caused by label leakage: `future_clicks_90d` contains information from the same future period used to construct `future_click_label`. Therefore, the leaked score is not a valid estimate of real-world predictive performance.

I removed `future_clicks_90d` from the final feature set and retained the honest ROC-AUC of **0.7512**.

### Final honest feature set

The final five features are:

1. `gsc_impressions`
2. `gsc_clicks`
3. `gsc_avg_position`
4. `ga4_sessions`
5. `sessions_ai`

The deliberately leaked `future_clicks_90d` feature was removed from the final feature set.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named limitation

**Uneven source coverage:** The main limitation of this slice is that GSC and GA4 availability differs across observations, so feature coverage is not uniform across all client-content pairs.

March rows:              9,841,378
GSC available:           3,611,061
GA4 sessions non-null:   6,822,637
AI sessions non-null:    6,822,637

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Label

- `future_click_label` — a binary future outcome indicating whether the client-content pair received at least one click during the future window from **2026-04-02 through 2026-06-30**. The label is not available at the March 2026 decision moment.

The observed label distribution in the matched dataset was 94,494 false and 39,358 true outcomes.

- `future_clicks_90d` — excluded from the honest feature set because it is derived from the future outcome window and directly leaks information about the label.